# Stage 1 f04 + Choices Inference (T4)

Thin Colab orchestration notebook for eval-only `x11_f04_candidate_choices` using the trained `f04_candidate_yes_no` artifact on a cheaper T4 runtime.


## 1. Mount Google Drive

Mount Google Drive before running the repo bootstrap flow.


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 2. Configure Paths

Set the repo checkout path, Drive output directory, trained `f04` artifact path, and optional data override.


In [2]:
from pathlib import Path

REPO_URL = "https://github.com/Demetri65/dl-kaggle-competition-final.git"
REPO_REF = "main"
REPO_DIR = Path("/content/dl-kaggle-competition-final")
SOURCE_ENV = Path("/content/.env")
UPLOADER_KEY = "_env_uploader"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/p2p_runs/stage1_f04_choices_t4"
DATA_DIR_OVERRIDE = ""
EXPERIMENT_ID = "x11_f04_candidate_choices"
EVAL_BATCH_SIZE = 4
INFERENCE_COMPLETION_BATCH_SIZE = 8
F04_ARTIFACT_DIR = "/content/drive/MyDrive/p2p_runs/notebook_c_a100/full_f04/f04_candidate_yes_no_seed42/model"
ARTIFACT_SEARCH_ROOT = "/content/drive/MyDrive"


## 3. Prepare The Repo

Upload `.env` if needed, sync the repo, and run `scripts/bootstrap_colab.sh`.


In [4]:
# -- 0. Colab setup ------------------------------------------------
import subprocess
import ipywidgets as widgets
from IPython.display import display


def get_uploaded_file(uploader):
    value = uploader.value

    if isinstance(value, dict):
        filename, uploaded_file = next(iter(value.items()))
        if isinstance(uploaded_file, dict):
            content = uploaded_file.get("content", uploaded_file.get("data"))
        else:
            content = uploaded_file
    else:
        uploaded_file = value[0]
        if isinstance(uploaded_file, dict):
            filename = uploaded_file["name"]
            content = uploaded_file["content"]
        else:
            filename = uploaded_file.name
            content = uploaded_file.content

    payload = content.tobytes() if hasattr(content, "tobytes") else bytes(content)
    return filename, payload


ready_to_bootstrap = SOURCE_ENV.exists()

if ready_to_bootstrap:
    print(f"Using existing {SOURCE_ENV}")
else:
    uploader = globals().get(UPLOADER_KEY)
    if uploader is None:
        uploader = widgets.FileUpload(accept=".env", multiple=False, description="Upload .env")
        globals()[UPLOADER_KEY] = uploader

    if not uploader.value:
        display(uploader)
        print("Select your local .env file in the upload widget above, then rerun this cell.")
    else:
        filename, payload = get_uploaded_file(uploader)
        SOURCE_ENV.write_bytes(payload)
        SOURCE_ENV.chmod(0o600)
        print(f"Saved {filename} to {SOURCE_ENV}")
        uploader.close()
        globals().pop(UPLOADER_KEY, None)
        ready_to_bootstrap = True

if ready_to_bootstrap:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)

    print(f"Repo synced to latest origin/{REPO_REF} at {REPO_DIR}")
    result = subprocess.run(
        ["bash", "scripts/bootstrap_colab.sh"],
        cwd=REPO_DIR,
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr, end="")
        raise RuntimeError(f"scripts/bootstrap_colab.sh failed with exit code {result.returncode}")


Saved .env to /content/.env
Repo synced to latest origin/main at /content/dl-kaggle-competition-final
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 33.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 65.5 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0

Using Kaggle CLI Kaggle CLI 2.0.1

Extracting pixels-to-predictions.zip
Competit

## 4. Helpers

Helpers resolve the trained `f04` artifact, run repo commands, and print resolved-config verification.


In [5]:
import json
import os
import shlex
import subprocess
from pathlib import Path

import yaml

REPO_ROOT = REPO_DIR.resolve()
Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT)

COMMON_OVERRIDES = []
if DATA_DIR_OVERRIDE:
    COMMON_OVERRIDES.append(f"data.data_dir={DATA_DIR_OVERRIDE}")


def with_common_overrides(overrides):
    return [*COMMON_OVERRIDES, *overrides]


def extend_with_overrides(args, overrides):
    for override in overrides:
        args.extend(["--set", override])


def run_repo_command(args):
    command = ["python3", *args]
    print("$", " ".join(shlex.quote(part) for part in command))
    result = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result


def resolve_saved_artifact_dir(explicit_dir: str, search_root: str, experiment_hint: str = "f04") -> Path:
    if explicit_dir:
        artifact_dir = Path(explicit_dir)
        if artifact_dir.name == "model":
            artifact_dir = artifact_dir.parent
        if artifact_dir.exists() and (artifact_dir / "model").exists() and (artifact_dir / "processor").exists():
            return artifact_dir
        print(f"Explicit artifact dir was not usable, falling back to auto-discovery: {artifact_dir}")

    root = Path(search_root)
    if not root.exists():
        raise FileNotFoundError(f"Artifact search root not found: {root}")

    candidates = []
    for marker_name in ("adapter_model.safetensors", "model_adapters.safetensors", "adapter_config.json"):
        for marker_path in root.rglob(marker_name):
            if marker_path.parent.name != "model":
                continue
            run_dir = marker_path.parent.parent
            if (run_dir / "model").exists() and (run_dir / "processor").exists():
                candidates.append(run_dir)

    unique_candidates = sorted({candidate.resolve() for candidate in candidates})
    hinted_candidates = [candidate for candidate in unique_candidates if experiment_hint in str(candidate).lower()]
    if len(hinted_candidates) == 1:
        return hinted_candidates[0]
    if not hinted_candidates and len(unique_candidates) == 1:
        return unique_candidates[0]

    candidate_list = hinted_candidates or unique_candidates
    if len(candidate_list) > 1:
        candidate_list = sorted(
            candidate_list,
            key=lambda candidate: max(path.stat().st_mtime for path in candidate.rglob("*") if path.exists()),
            reverse=True,
        )
        print(f"Auto-discovery found multiple candidates; using the most recently updated one: {candidate_list[0]}")
        return candidate_list[0]

    formatted = "\n".join(str(candidate) for candidate in candidate_list[:20]) or "<none found>"
    raise RuntimeError(
        "Could not resolve a saved artifact dir automatically. "
        f"Set F04_ARTIFACT_DIR explicitly. Candidates:\n{formatted}"
    )


def verify_resolved_config(run_root: Path) -> None:
    candidates = sorted(run_root.glob(f"{EXPERIMENT_ID}_seed*/resolved_config.yaml"))
    if not candidates:
        print(f"Resolved config not found under: {run_root}")
        return
    resolved_config_path = candidates[-1]
    config = yaml.safe_load(resolved_config_path.read_text())
    enabled_fields = [name for name, enabled in config["fields"].items() if enabled]
    payload = {
        "resolved_config_path": str(resolved_config_path),
        "training.epochs": config["training"]["epochs"],
        "training.eval_batch_size": config["training"]["eval_batch_size"],
        "training.bf16": config["training"]["bf16"],
        "training.fp16": config["training"]["fp16"],
        "scoring.max_completion_batch_size": config["scoring"]["max_completion_batch_size"],
        "enabled_fields": enabled_fields,
        "eval_artifact_dir": config["runtime"]["eval_artifact_dir"],
    }
    print(json.dumps(payload, indent=2, sort_keys=True))


RESOLVED_F04_ARTIFACT_DIR = resolve_saved_artifact_dir(F04_ARTIFACT_DIR, ARTIFACT_SEARCH_ROOT, experiment_hint="f04")

print(f"Repo root: {REPO_ROOT}")
print(f"Drive output dir: {DRIVE_OUTPUT_DIR}")
print(f"Resolved f04 artifact dir: {RESOLVED_F04_ARTIFACT_DIR}")
if DATA_DIR_OVERRIDE:
    print(f"Data override: {DATA_DIR_OVERRIDE}")


Repo root: /content/dl-kaggle-competition-final
Drive output dir: /content/drive/MyDrive/p2p_runs/stage1_f04_choices_t4
Resolved f04 artifact dir: /content/drive/MyDrive/p2p_runs/notebook_c_a100/full_f04/f04_candidate_yes_no_seed42


## 6. Full Validation Eval

Run full validation with the same eval-only artifact-backed settings. This does not retrain.


In [ ]:
full_root = Path(DRIVE_OUTPUT_DIR) / "full"
full_args = [
    "scripts/run_experiment.py",
    "--experiment", EXPERIMENT_ID,
    "--output-dir", str(full_root),
]
extend_with_overrides(full_args, with_common_overrides([
    "training.epochs=0",
    "training.bf16=false",
    "training.fp16=true",
    f"training.eval_batch_size={EVAL_BATCH_SIZE}",
    f"scoring.max_completion_batch_size={INFERENCE_COMPLETION_BATCH_SIZE}",
    f"runtime.eval_artifact_dir={RESOLVED_F04_ARTIFACT_DIR}",
]))
run_repo_command(full_args)
verify_resolved_config(full_root)


$ python3 scripts/run_experiment.py --experiment x11_f04_candidate_choices --output-dir /content/drive/MyDrive/p2p_runs/stage1_f04_choices_t4/full --set training.epochs=0 --set training.bf16=false --set training.fp16=true --set training.eval_batch_size=4 --set scoring.max_completion_batch_size=8 --set runtime.eval_artifact_dir=/content/drive/MyDrive/p2p_runs/notebook_c_a100/full_f04/f04_candidate_yes_no_seed42


## 7. Summarize Results

Print the top validation runs from `results/experiments.csv`.


In [ ]:
summary_args = [
    "scripts/summarize_results.py",
    "--sort-by", "val_accuracy",
    "--top", "20",
]
run_repo_command(summary_args)
